# Train CenterNet HRNet-W18 on COCO + NIH + Lumos

This notebook trains a new CenterNet model on BUU, Mendeley, MICCAI, patient-grouped NIH ChestX-ray14, and Lumos AP annotations. Training runs on fast Colab storage; essential checkpoints are copied to Google Drive after every epoch.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import shutil

DRIVE_PACKAGE = Path('/content/drive/MyDrive/SOS Colab')
PROJECT_ROOT = Path('/content/Spine-Opportunistic-Screening')
LOCAL_RUN_PARENT = Path('/content/centernet_runs')
DRIVE_RUN_PARENT = Path('/content/drive/MyDrive/spine_centernet_runs_nih_lumos')
BACKBONE = 'hrnet_w18'
EXPERIMENT = f'centernet_{BACKBONE}_coco_nih_lumos'

RESUME = False  # Keep False for the first run; enable only to continue this exact experiment.
RUN_OVERFIT_SMOKE = False
RUN_TEST_EVALUATION = False
VALIDATION_CHECKPOINT = 'best_usable_recall'
BATCH_SIZE = 2
NUM_WORKERS = 2
EPOCHS = 80

if not DRIVE_PACKAGE.is_dir():
    raise FileNotFoundError(f'Upload the SOS Colab folder to {DRIVE_PACKAGE}')
if PROJECT_ROOT.exists():
    shutil.rmtree(PROJECT_ROOT)
shutil.copytree(DRIVE_PACKAGE, PROJECT_ROOT)
%cd /content/Spine-Opportunistic-Screening
print('project:', PROJECT_ROOT)
print('backbone:', BACKBONE)
print('experiment:', EXPERIMENT)

In [ ]:
!pip install -q -r requirement.txt
import platform, cv2, numpy as np, torch, timm
print('python', platform.python_version())
print('torch', torch.__version__, 'cuda', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('timm', timm.__version__, 'opencv', cv2.__version__, 'numpy', np.__version__)
if not torch.cuda.is_available():
    raise RuntimeError('Select a GPU runtime before training.')

## Verify the merged dataset

These checks fail before training if counts, sources, the dataset fingerprint, or an image path differs from the reviewed package.

In [ ]:
import json
from collections import Counter

EXPECTED = {
    'train': {'images': 1788, 'annotations': 22076, 'nih_images': 257, 'lumos_images': 128},
    'val': {'images': 223, 'annotations': 2976, 'nih_images': 32, 'lumos_images': 16},
    'test': {'images': 223, 'annotations': 3060, 'nih_images': 32, 'lumos_images': 16},
}

for split, expected in EXPECTED.items():
    split_dir = Path('dataset') / split
    with (split_dir / '_annotations.keypoints.coco.json').open(encoding='utf-8') as file:
        coco = json.load(file)
    nih_images = [item for item in coco['images'] if item['file_name'].startswith('images/nih_chestxray14/')]
    lumos_images = [item for item in coco['images'] if item['file_name'].startswith('images/lumos_ap/')]
    assert len(coco['images']) == expected['images'], (split, len(coco['images']))
    assert len(coco['annotations']) == expected['annotations'], (split, len(coco['annotations']))
    assert len(nih_images) == expected['nih_images'], (split, len(nih_images))
    assert len(lumos_images) == expected['lumos_images'], (split, len(lumos_images))
    missing = [item['file_name'] for item in coco['images'] if not (split_dir / item['file_name']).is_file()]
    assert not missing, f'{split}: {len(missing)} missing files; first={missing[:1]}'
    sources = Counter(item.get('source_dataset', 'unknown') for item in coco['images'])
    print(split, len(coco['images']), 'images |', len(coco['annotations']), 'annotations |', dict(sources))

with Path('dataset/split_summary.json').open(encoding='utf-8') as file:
    split_summary = json.load(file)
assert split_summary['version'] == 'coco_nih_lumos'
print('corner order:', split_summary['corner_order'])

from src.data.dataset_provenance import build_dataset_provenance
EXPECTED_FINGERPRINT = 'd99fbd046ece0cd2d10598fa1fb1da0eed24cb8542270f54a732616c8cb9e8f3'
dataset_provenance = build_dataset_provenance('dataset')
assert dataset_provenance['fingerprint'] == EXPECTED_FINGERPRINT, dataset_provenance['fingerprint']
print('dataset fingerprint:', dataset_provenance['fingerprint'])

In [ ]:
import subprocess, sys

for split in ('train', 'val', 'test'):
    command = [
        sys.executable, '-u', '-m', 'src.workflows.check_centernet_dataset',
        '--dataset-root', 'dataset', '--split', split,
        '--image-size', '1024', '--backbone', BACKBONE,
        '--batch-size', '2', '--num-workers', '0', '--limit', '4',
    ]
    if split == 'train':
        command.append('--model-forward')
    subprocess.run(command, check=True)

## Preview NIH and Lumos training targets

The yellow quadrilaterals below are annotations after the deterministic resize/pad transform, in TL, TR, BL, BR order.

In [ ]:
import matplotlib.pyplot as plt
from src.data.centernet_dataset import CenterNetCocoDataset

preview_dataset = CenterNetCocoDataset('dataset', 'train', image_size=1024, augment=False)
preview_groups = {
    'NIH': [index for index, record in enumerate(preview_dataset.samples) if record['image']['file_name'].startswith('images/nih_chestxray14/')][:4],
    'Lumos': [index for index, record in enumerate(preview_dataset.samples) if record['image']['file_name'].startswith('images/lumos_ap/')][:4],
}
assert all(len(indices) == 4 for indices in preview_groups.values())
fig, axes = plt.subplots(2, 4, figsize=(20, 12))
for axis_row, (source_name, indices) in zip(axes, preview_groups.items()):
    for axis, index in zip(axis_row, indices):
        sample = preview_dataset[index]
        image = np.clip(sample['input'].numpy().transpose(1, 2, 0) + 0.5, 0, 1)
        axis.imshow(image)
        for corners in sample['gt_corners'][:int(sample['gt_count'])].numpy():
            polygon = corners[[0, 1, 3, 2, 0]]
            axis.plot(polygon[:, 0], polygon[:, 1], 'y-', linewidth=1)
        axis.set_title(f'{source_name}: {Path(sample["file_name"]).name}')
        axis.axis('off')
plt.tight_layout()

## Optional overfit smoke

Enable RUN_OVERFIT_SMOKE in the settings cell for the first package check. This is a short plumbing test, not a model-quality result.

In [ ]:
if RUN_OVERFIT_SMOKE:
    subprocess.run([
        sys.executable, '-u', '-m', 'src.train_centernet',
        '--dataset-root', 'dataset', '--output-dir', '/content/centernet_overfit',
        '--experiment-name', f'{BACKBONE}_nih_lumos_overfit_8', '--backbone', BACKBONE,
        '--input-size', '512', '--overfit-samples', '8', '--epochs', '10',
        '--batch-size', '2', '--num-workers', '0', '--lr', '3e-4',
        '--wh-weight', '0.5', '--peak-thresh', '0.10', '--eval-topk', '50',
        '--progress-every', '5', '--amp',
    ], check=True)
else:
    print('optional overfit smoke skipped')

## Restore and train

For the first run, keep RESUME=False. After this exact experiment has produced last.pt, set RESUME=True to continue it. Dataset fingerprint validation prevents accidentally resuming an older run. Training uses local Colab storage and copies last.pt, all four best checkpoints, the training/landmark logs, and validation artifacts to Drive after every epoch.

In [ ]:
local_run = LOCAL_RUN_PARENT / EXPERIMENT
drive_run = DRIVE_RUN_PARENT / EXPERIMENT
if RESUME and drive_run.is_dir():
    local_run.mkdir(parents=True, exist_ok=True)
    shutil.copytree(drive_run, local_run, dirs_exist_ok=True)
    print('restored:', drive_run)

command = [
    sys.executable, '-u', '-m', 'src.train_centernet',
    '--config', 'configs/config.yaml', '--dataset-root', 'dataset',
    '--output-dir', str(LOCAL_RUN_PARENT), '--experiment-name', EXPERIMENT,
    '--backup-dir', str(DRIVE_RUN_PARENT), '--backbone', BACKBONE,
    '--input-size', '1024', '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS),
    '--lr', '1e-4', '--weight-decay', '1e-4',
    '--hm-weight', '1.0', '--reg-weight', '1.0', '--wh-weight', '0.5',
    '--peak-thresh', '0.10', '--eval-topk', '50', '--amp',
    '--early-stop-patience', '15', '--early-stop-metric', 'center_f1_12px',
    '--save-preview-every', '5', '--preview-images', '4',
    '--progress-every', '25',
]
resume_checkpoint = local_run / 'last.pt'
if RESUME and resume_checkpoint.is_file():
    command += ['--resume-checkpoint', str(resume_checkpoint)]
print(' '.join(command))
subprocess.run(command, check=True)

## Evaluate the best validation checkpoint

This evaluates the checkpoint named by VALIDATION_CHECKPOINT (best usable recall by default) on validation. Landmark checkpoints are created only after the detection guardrails pass; the cell falls back to best-center-F1 if the selected checkpoint is unavailable. Test evaluation is guarded separately so it is not used repeatedly during tuning.

In [ ]:
best_checkpoint = local_run / f'{VALIDATION_CHECKPOINT}.pt'
if not best_checkpoint.is_file():
    print('selected checkpoint is unavailable; falling back to best_center_f1.pt')
    best_checkpoint = local_run / 'best_center_f1.pt'
if not best_checkpoint.is_file():
    raise FileNotFoundError(best_checkpoint)
val_output = local_run / f'evaluation_val_{best_checkpoint.stem}'
subprocess.run([
    sys.executable, '-u', '-m', 'src.evaluate_centernet',
    '--evaluation-profile', 'research',
    '--checkpoint', str(best_checkpoint), '--dataset-root', 'dataset',
    '--split', 'val', '--output-dir', str(val_output),
    '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS),
    '--peak-thresh', '0.10', '--topk', '50',
], check=True)
drive_val_output = drive_run / f'evaluation_val_{best_checkpoint.stem}'
drive_val_output.mkdir(parents=True, exist_ok=True)
for artifact in val_output.iterdir():
    if artifact.is_file():
        temporary = drive_val_output / (artifact.name + '.tmp')
        shutil.copy2(artifact, temporary)
        temporary.replace(drive_val_output / artifact.name)
print('validation artifacts synced to', drive_val_output)

In [ ]:
import pandas as pd

per_image = pd.read_csv(val_output / 'per_image_metrics.csv')
rows = []
for (model_name, source_name), group in per_image.groupby(['model', 'source_dataset']):
    matched = group['matched_12px'].sum()
    precision = matched / max(group['pred_count'].sum(), 1)
    recall = matched / max(group['gt_count'].sum(), 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    rows.append({
        'model': model_name, 'source': source_name, 'images': len(group),
        'f1_12px': f1, 'recall_12px': recall,
        'mean_corner_mae_12px': group['corner_mae_12px'].mean(),
        'mean_count_error': group['count_error'].mean(),
    })
source_metrics = pd.DataFrame(rows).sort_values(['model', 'source'])
display(source_metrics)
source_metrics.to_csv(val_output / 'per_source_metrics.csv', index=False)
shutil.copy2(val_output / 'per_source_metrics.csv', drive_val_output / 'per_source_metrics.csv')

In [ ]:
if RUN_TEST_EVALUATION:
    test_output = local_run / 'evaluation_test'
    subprocess.run([
        sys.executable, '-u', '-m', 'src.evaluate_centernet',
        '--evaluation-profile', 'research',
        '--checkpoint', str(best_checkpoint), '--dataset-root', 'dataset',
        '--split', 'test', '--output-dir', str(test_output),
        '--batch-size', str(BATCH_SIZE), '--num-workers', str(NUM_WORKERS),
        '--peak-thresh', '0.10', '--topk', '50',
    ], check=True)
    shutil.copytree(test_output, drive_run / 'evaluation_test', dirs_exist_ok=True)
else:
    print('test evaluation is disabled; enable it only after the training setup is frozen')